In [1]:
import pandas as pd

In [3]:
df = pd.read_json('../data/인터뷰/ckmk_d_bm_f_n_169087.json')
df

,version,dataSet,rawDataInfo
info,1,"{'date': '20230116', 'occupation': 'BM', 'chan...",NaN
question,1,{'raw': {'text': '지원자님께서 가지고 계신 나만의 스트레스 해소법이 ...,"{'fileFormat': 'wav', 'fileSize': 664398, 'dur..."
answer,1,{'raw': {'text': '저는 저 혼자 시간을 보내는 것을 우선적으로 하고요...,"{'fileFormat': 'wav', 'fileSize': 1900238, 'du..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, info to answer
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   version      3 non-null      int64 
 1   dataSet      3 non-null      object
 2   rawDataInfo  2 non-null      object
dtypes: int64(1), object(2)
memory usage: 96.0+ bytes


In [5]:
# read_json을 통해 json파일을 로드했는데? 데이터 형태가 데이터프레임에 최적화x
# 답변의 데이터, 답변의 요약데이터를 추출 -> kobart 모델을 이용해서 요약 학습
# json 파일을 dict형태로 변환 -> 데이터 추출
import json

In [6]:
# 파일을 로드해서 dict형태로 변환
with open('../data/인터뷰/ckmk_d_bm_f_n_169087.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [8]:
type(data)

dict

In [9]:
from pprint import pprint

In [11]:
pprint(data)

{'dataSet': {'answer': {'emotion': [],
                        'intent': [{'category': 'attitude',
                                    'expression': '',
                                    'text': ''}],
                        'raw': {'text': '저는 저 혼자 시간을 보내는 것을 우선적으로 하고요. 예를 들어서 '
                                        '혼자 산책을 한다든지 아니면 혼자 카페를 간다든지 아니면 혼자 코인 '
                                        '노래방을 간다든지 해서 스트레스를 해소하는 편입니다. 물론 남들과 '
                                        '만나면서 시간을 보내는 것도 굉장히 좋아하지만요. 아무래도 내 '
                                        '혼자만의 시간이 있어야 어느 정도 생각 정리도 되고 그러면서 내 '
                                        '안에 있던 응어리도 절로 풀러지는 그런 것을 경험하게 되었습니다. '
                                        '그리고 코인 노래방이나 이런 곳에 가면은 아무래도 제가 노래 부르는 '
                                        '것도 좋아하고 그리고 노래를 부르면서 그런 소리가 웅웅거리고 또 '
                                        '소리를 지른다는 거에 대해서 해소감도 있는 것 같고요. 그래서 저는 '
                                        '일단 혼자만의 시간을 보내는데 뭐 혼자서 노래방을 가거나 아니면 '

In [25]:
data['dataSet']['answer']['raw']['text']

'저는 저 혼자 시간을 보내는 것을 우선적으로 하고요. 예를 들어서 혼자 산책을 한다든지 아니면 혼자 카페를 간다든지 아니면 혼자 코인 노래방을 간다든지 해서 스트레스를 해소하는 편입니다. 물론 남들과 만나면서 시간을 보내는 것도 굉장히 좋아하지만요. 아무래도 내 혼자만의 시간이 있어야 어느 정도 생각 정리도 되고 그러면서 내 안에 있던 응어리도 절로 풀러지는 그런 것을 경험하게 되었습니다. 그리고 코인 노래방이나 이런 곳에 가면은 아무래도 제가 노래 부르는 것도 좋아하고 그리고 노래를 부르면서 그런 소리가 웅웅거리고 또 소리를 지른다는 거에 대해서 해소감도 있는 것 같고요. 그래서 저는 일단 혼자만의 시간을 보내는데 뭐 혼자서 노래방을 가거나 아니면 혼자서 카페를 가거나 그런 식으로 스트레스를 해소한다 라고 말씀을 드릴 수 있을 것 같습니다.'

### kobart 복습
1. 인터뷰 폴더 안에 있는 json 파일들을 로드
    - 답변에 대한 데이터들을 하나의 리스트로 생성
    - 답변 요약에 대한 데이터들을 하나의 리스트로 생성
2. 위에서 생성된 데이터들 중 train 데이터는 처음부터 100번째 데이터까지
3. 마지막에서 10번째부터 마지막까지의 데이터를 validation 데이터로 사용
4. train, test 데이터를 DatasetDict 형태로 변환
5. tokenizer와 model을 로드하여 데이터를 학습하고 검증
    - input의 최대 사이즈 : 512
    - output의 최대 사이즈 : 256
6. test 데이터를 생성 : 원본의 리스트에서 200번째의 데이터를 이용하여 요약을 생성

In [14]:
from glob import glob

In [46]:
import numpy as np
from datasets import Dataset, DatasetDict
# 문장 간의 검증 지표를 만들어주는 라이브러리
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
glob('../data/인터뷰/*.json')

['../data/인터뷰\\ckmk_d_bm_f_n_169087.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169088.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169393.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169394.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169395.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169443.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169453.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169454.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169455.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169456.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169513.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169553.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169554.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169555.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169556.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169557.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169559.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169560.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169561.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_169562.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_170083.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_170084.json',
 '../data/인터뷰\\ckmk_d_bm_f_n_170085.json',
 '../data/인

In [16]:
file_list = glob('../data/인터뷰/*.json')

In [34]:
texts = []
summary = []

for file_name in file_list:
    with open(file_name, 'r', encoding='utf-8') as f:
        data2 = json.load(f, strict=False)
        text = data2['dataSet']['answer']['raw']['text']
        summ = data2['dataSet']['answer']['raw']['text']
        texts.append(text)
        summary.append(summ)


In [35]:
texts

['저는 저 혼자 시간을 보내는 것을 우선적으로 하고요. 예를 들어서 혼자 산책을 한다든지 아니면 혼자 카페를 간다든지 아니면 혼자 코인 노래방을 간다든지 해서 스트레스를 해소하는 편입니다. 물론 남들과 만나면서 시간을 보내는 것도 굉장히 좋아하지만요. 아무래도 내 혼자만의 시간이 있어야 어느 정도 생각 정리도 되고 그러면서 내 안에 있던 응어리도 절로 풀러지는 그런 것을 경험하게 되었습니다. 그리고 코인 노래방이나 이런 곳에 가면은 아무래도 제가 노래 부르는 것도 좋아하고 그리고 노래를 부르면서 그런 소리가 웅웅거리고 또 소리를 지른다는 거에 대해서 해소감도 있는 것 같고요. 그래서 저는 일단 혼자만의 시간을 보내는데 뭐 혼자서 노래방을 가거나 아니면 혼자서 카페를 가거나 그런 식으로 스트레스를 해소한다 라고 말씀을 드릴 수 있을 것 같습니다.',
 '네 여러 관계자와 협력을 해서 네 성공적으로 일정을 수행해 본 적이 있었고요. 아무래도 사람이 많다 보니까 저의 생각을 먼저 고수하는 것보다는 그들의 의견을 먼저 듣는 것에서 부터 시작했습니다. 아무래도 서로가 뭐라고 해야 될까요. 각자 보이는 생각이 다 맞다고 우기면은 거기서 더 잘 나아갈 수가 없더라고요. 그래서 들어봤을 때 이 사람의 이야기를 먼저 듣고 만약에 이게 아니다 라는 느낌이 들었을 때는 제가 이 사람 말에 우선 이해와 공감을 먼저 했습니다. 이러한 상황에서 당신의 말은 이렇군요 저도 그럴 것 같습니다 하지만 저희는 이런 결과물로 내야 되는데 여기에서 이 상황만 생각하면 조금은 어려울 것 같아요. 그래서 우리가 이런 비 상황도 함께 고려해서 좋게 결과를 만들어 봅시다 라는 식으로 조금은 애둘러서 하지만 강하게 말을 했었고 그로 인해서 협력이 잘 되었던 기억이 납니다. 네 그런 식으로 진행을 해서 굉장히 잘 수행되었다고 말씀드릴 수 있을 것 같습니다.',
 '대부분은 왠지 전사나 마법사를 고를 것 같은데요. 저는 조금은 독특하게 힐러 역할을 해보고 싶습니다. 힐러는 어떻게 보면은 굉장히 소위 게임하

In [68]:
len(texts)

12804

In [44]:
train_docs = texts[:100]
train_sums = summary[:100]
valid_docs = texts[-10:]
valid_sums = summary[-10:]

In [47]:
raw_ds = DatasetDict(
    {
        'train' : Dataset.from_dict(
            {
                'document' : train_docs,
                'summary' : train_sums
             }
        ),
        'validation' : Dataset.from_dict(
            {
                'document' : valid_docs,
                'summary' : valid_sums
            }
        )
    }
)

In [49]:
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 10
    })
})

In [51]:
model_name = 'gogamza/kobart-base-v2'

In [52]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 입력 / 출력 문장의 최대 길이 설정
max_input_len = 512
max_target_len = 256

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


In [53]:
def tok_fn(batch):

    inputs = tokenizer(
        batch['document'],
        max_length = max_input_len,
        padding = 'max_length',       # 고정길이 패딩 사용
        truncation = True             # 최대 길이보다 긴 경우 자른다
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch['summary'],
            max_length = max_target_len,
            padding = 'max_length',
            truncation = True
        )

    labels_ids = np.array(labels['input_ids'])

    labels_ids[labels_ids == tokenizer.pad_token_id] = -100

    inputs['labels'] = labels_ids.tolist()

    return inputs

In [54]:
tokenized_ds = raw_ds.map(tok_fn, batched=True, remove_columns=['document', 'summary'])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 10/10 [00:00<00:00, 615.49 examples/s]


In [55]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10
    })
})

In [56]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [57]:
rouge = evaluate.load('rouge')

In [58]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # vocab -> ['A', 'B', 'C']
    # decoder -> preds = [13, 17, 102, ...] -> 
    # vocab 인덱스로 구성된 리스트를 다시 문자의 형태로 변환하기 위해서 padding 토큰은 무시하기 위해 -100으로 구성 -> 원래의 padding 토큰의 id값으로 전환
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )
    # 텍스트 디코딩
    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 문장에서 좌우의 공백이 존재하는 경우 다른 값으로 측정하기 때문에 좌우 공백 제거
    pred_str = [doc.strip() for doc in pred_str]
    label_str = [doc.strip() for doc in label_str]

    # ROUGE 계산
    result = rouge.compute(
        predictions = pred_str,
        references = label_str,
        use_stemmer = True          # 점수 계산시 단어의 어간을 기준으로 비교하는 의미
    )
    
    # ROUGE를 보기 편한 형태로 변경
    result = {k : round(v * 100, 2) for k, v in result.items()}

    return result

In [59]:
args = Seq2SeqTrainingArguments(
    output_dir='./kobart',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    num_train_epochs=10,
    logging_steps=10,

    # generate 설정을 변경
    predict_with_generate=True,
    generation_max_length=70,
    # 요약 데이터를 생성할때 문장 후보의 탐색의 개수를 설정
    generation_num_beams=4,

    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    report_to=[]
)

In [60]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

C:\Users\student\AppData\Local\Temp\ipykernel_13108\1366852772.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.426200,0.165842,0.000000,0.000000,0.000000,0.000000
2,0.032800,0.017284,0.000000,0.000000,0.000000,0.000000
3,0.026100,0.019032,0.000000,0.000000,0.000000,0.000000
4,0.010500,0.015254,0.000000,0.000000,0.000000,0.000000
5,0.009100,0.017460,0.000000,0.000000,0.000000,0.000000
6,0.011600,0.011359,0.000000,0.000000,0.000000,0.000000
7,0.002000,0.009994,0.000000,0.000000,0.000000,0.000000
8,0.001700,0.028506,0.000000,0.000000,0.000000,0.000000
9,0.003500,0.032968,0.000000,0.000000,0.000000,0.000000
10,0.000700,0.032879,0.000000,0.000000,0.000000,0.000000


c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'forced_eos_token_id': 1}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: Use

TrainOutput(global_step=130, training_loss=0.042517339570734364, metrics={'train_runtime': 2039.0765, 'train_samples_per_second': 0.49, 'train_steps_per_second': 0.064, 'total_flos': 304868229120000.0, 'train_loss': 0.042517339570734364, 'epoch': 10.0})

In [74]:
test_text = texts[200]

inputs = tokenizer(test_text, return_tensors='pt', truncation=True, max_length=max_input_len)
inputs.pop('token_type_ids', None)

gen_ids = model.generate(
    **inputs.to(model.device),
    max_length = 20,
    num_beams = 4,
    do_sample=False
)

print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))

네 저는 지금 백지연이 지은 크리티컬 매스 라는 책을 요즘 읽고 있는데요. 제가


In [75]:
test_text

'네 저는 지금 백지연이 지은 크리티컬 매스 라는 책을 요즘 읽고 있는데요. 제가 백지연을 예전에는 제가 굉장히 존경하기도 했었는데 소통 능력이 굉장히 뛰어나다 라는 생각이 많이 들었었고 그리고 다양한 사람들과 인터뷰 경력이 있으시잖아요. 그래서 타인을 이해할 수 있는 폭이 넓지 않을까 라는 생각에서 이 책을 읽게 되었고 그리고 이 책에서 느낀 게 제가 정말 남들이 감동시킬 만한 노력을 한다면 그것은 타인들이 다 알아줄 것이다 라는 이야기가 있어요. 그래서 정말 노력을 해서 어떤 재능이든 개발할 수 있겠구나 라는 생각이 들었고 여기에 많은 분들을 인터뷰한 그런 내용이 있습니다. 그래서 저도 예전에 다 다양한 사람들의 이야기를 들으면서 저도 많이 발전이 됐었는데 타인의 이야기에서 제 장점을 또 끌어내고 그것을 할 수 있는 그런 깨달음을 제가 얻게 되어서 이 책을 굉장히 감명 깊게 읽고 있고요. 그리고 이 책을 읽으면서 인터뷰를 하고 다른 사람의 이야기를 듣는 법에 대해서도 깨닫게 된 것 같습니다.'